<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Prompt Engineering for Unsupervised Audio Classification Using CLAP</b></h1>
</div>

## Requirements and Approach

This document defines the engineering specification for a reproducible zero-shot environmental-sound classification study using LAION-CLAP and ESC-50.

The design isolates **prompt formulation** as the controlled experimental variable. Dataset, candidate labels, pretrained checkpoint, embedding normalization, similarity computation, and evaluation procedure remain fixed across strategies. The workflow emphasizes deterministic data handling, reusable inference functions, traceable metrics, explicit output generation, and separation between locally recomputed results and external benchmark references.

## Global Requirements

| Requirement | Implemented choice |
| --- | --- |
| Dataset | ESC-50 |
| Evaluation population | 2,000 clips / 50 classes |
| Model | pretrained LAION-CLAP |
| Learning regime | zero-shot |
| ESC-50 fine-tuning | none |
| Prompt strategies | 10 |
| Baseline | class label only |
| Embedding normalization | L2 |
| Similarity | normalized dot product / cosine similarity |
| Best-strategy criterion | measured Top-1 accuracy |
| Ranking analysis | Top-1 through Top-10 |
| Class diagnostics | per-class accuracy |
| Metadata analysis | Pearson + Spearman |
| Outputs | CSV tables + report figures |
| External benchmarks | contextual only; not recomputed unless explicitly marked |

## 1. Data and Output Paths

**Approach:** Use repository-relative `Path` objects for `data/`, `outputs/tables/`, and `outputs/figures/`; create missing output directories before any read/write operation.

**Acceptance:** All three directories resolve successfully and the notebook reports the paths used for the run.


## 2. ESC-50 Dataset Preparation

**Approach:** Check for a local ESC-50 extraction first. If absent, download the official archive, extract it under `data/`, remove the temporary ZIP, and validate both the audio directory and metadata CSV.

**Acceptance:** `ESC-50-master/audio/` and `ESC-50-master/meta/esc50.csv` both exist before downstream execution.


## 3. Dataset Exploration

**Approach:** Load the ESC-50 metadata with pandas, derive a deterministic sorted class list, and export a compact metadata preview plus the canonical class list.

**Acceptance:** The evaluation set contains 2,000 samples and exactly 50 unique classes, with a stable class ordering reused everywhere else.


## 4. CLAP Model Initialization

**Approach:** Select CUDA when available and otherwise CPU; instantiate one pretrained LAION-CLAP module, load its checkpoint once, move it to the selected device, and switch to evaluation mode.

**Acceptance:** The model loads without error, the device is reported, and the same model instance is reused for every prompt strategy.


## 5. Prompt Strategy Definition

**Approach:** Store the ten approved prompt templates in one explicit mapping, including the class-only baseline. Clean ESC-50 labels consistently before template substitution.

**Acceptance:** Exactly ten strategies are available and every strategy operates on the same 50 semantic class labels.


## 6. Zero-Shot Classification Framework

**Approach:** Implement reusable functions for class-name cleaning, L2 normalization, text embedding, audio embedding, and prompt evaluation. Compute similarity from normalized embeddings and preserve ranked Top-k predictions.

**Acceptance:** Every evaluated sample yields a finite similarity vector, one Top-1 prediction, correctness status, confidence information, and ranked candidates through Top-10.


## 7. Full Prompt Benchmark

**Approach:** Run the same evaluation function over the complete ESC-50 set for each of the ten prompt templates and record accuracy plus runtime information.

**Acceptance:** A ranked benchmark table is produced for all ten strategies and exported to `outputs/tables/`.


## 8. Best Prompt and Class-wise Accuracy

**Approach:** Select the best strategy from the measured benchmark, retain its sample-level predictions, and aggregate those predictions by true class.

**Acceptance:** The best strategy is data-driven, its prediction table is exported, and class-wise accuracy is available for all 50 classes.


## 9. Prompt Strategy Comparison

**Approach:** Map internal strategy identifiers to report-ready names and derive a concise comparison directly from the benchmark dataframe.

**Acceptance:** All ten strategies, including the class-only baseline, appear with measured Top-1 accuracy in both CSV and figure outputs.


## 10. Class-wise Accuracy Summary

**Approach:** Extract the ten lowest- and ten highest-performing classes from the complete best-prompt class-wise table and convert labels to readable form.

**Acceptance:** The summary remains traceable to the 50-class result and is exported as both CSV and diagnostic figure.


## 11. Main Benchmark Summary

**Approach:** Derive the best-prompt accuracy, class-only baseline, and absolute percentage-point improvement from the benchmark variables.

**Acceptance:** The summary reports the retained strategy/template and computes the improvement without manually duplicating performance values.


## 12. Top-k Accuracy

**Approach:** Use the stored ranked predictions for the best prompt to compute Top-k accuracy for every k from 1 through 10, then export table and line visualizations.

**Acceptance:** Top-k values are finite, bounded in [0,100]%, non-decreasing with k, and saved to the project outputs.


## 13. AudioSet Metadata Analysis

**Approach:** Join the curated ESC-50-to-AudioSet mapping with class-wise accuracy, transform video-count scale where required, and compute both Pearson and Spearman associations.

**Acceptance:** All 50 ESC-50 classes are represented and the merged metadata table plus correlation summary and figures are exported.


## 14. Comparison with Published ESC-50 Classifiers

**Approach:** Represent external benchmark values in a structured table that records method, accuracy, supervision type, provenance, and whether the value was recomputed. Insert the CLAP value from the current experiment.

**Acceptance:** The project result is clearly separated from external supervised/fine-tuned references and no external value is presented as if recomputed locally.


## 15. Results Summary

**Approach:** Build the final summary table from live benchmark and Top-k objects after all upstream calculations are complete.

**Acceptance:** The exported summary contains the retained prompt, baseline, percentage-point improvement, and representative Top-k metrics with no hard-coded principal result.


## Requirement-to-Code Traceability

| Requirement block | Implementation evidence |
| --- | --- |
| Paths and dataset integrity | Tasks 1–3 |
| Fixed CLAP inference environment | Task 4 |
| Controlled prompt definitions | Task 5 |
| Reusable zero-shot scoring pipeline | Task 6 |
| Complete prompt benchmark | Task 7 |
| Best-prompt and class-level analysis | Tasks 8–10 |
| Benchmark gain and ranking quality | Tasks 11–12 |
| AudioSet metadata investigation | Task 13 |
| External benchmark context | Task 14 |
| Final computed study summary | Task 15 |

The implementation is considered compliant only when every mapped task executes successfully and the reported evidence is generated from the same run.